# PMSA v2 — Phase 1 extraction smoke test (Kaggle T4)

Goal: validate the `extract → cache → baseline` loop on a **small** GenImage subset
before scaling. Start with 2 generators × 2k images/class. If the UnivFD baseline
AUC looks sane (in-distribution should be high; leave-one-generator-out is the real
test), the pipeline works and you can scale `PER_CLASS_LIMIT`.

**Settings:** turn ON GPU and Internet in the notebook settings panel. Use a **T4**
(single or T4 ×2) — **not the P100**: Kaggle's PyTorch dropped support for the
P100's sm_60 compute capability, so it warns and may fail. The T4 is sm_75 and runs
clean. The pipeline uses a single GPU, so T4 ×2 just leaves one card idle (harmless).
Add the GenImage dataset to the notebook's inputs.

In [4]:
# 1) Get the code. IDEMPOTENT: always operate on the absolute repo path, so
#    re-running this cell pulls in place and never nests pmsa_v2/pmsa_v2/...
import os
os.chdir('/kaggle/working')
REPO = '/kaggle/working/pmsa_v2'
if os.path.isdir(REPO + '/.git'):
    !cd {REPO} && git pull -q
else:
    !git clone -q https://github.com/Y-ash-Y/AI-Detection-.git {REPO}
os.chdir(REPO)
print('cwd:', os.getcwd())
!pip install -q -e . transformers --no-deps 2>/dev/null; pip install -q pyyaml joblib scikit-learn
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


cwd: /kaggle/working/pmsa_v2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pmsa (pyproject.toml) ... done
cuda: True Tesla T4


In [ ]:
# 2) Auto-discover the GenImage layout. Add a dataset first (recommended:
#    'yangsangtai/tiny-genimage' — small, multi-generator, enables LOGO).
#    GenImage layout: <root>/<generator>/<split>/{ai,nature}/*  (ai=fake, nature=real)
import os, glob
print('inputs mounted:', sorted(glob.glob('/kaggle/input/*')) or 'NONE — add a dataset!')

# find every fake ('ai') leaf folder, then derive root/generator/split from it
ai_dirs = glob.glob('/kaggle/input/**/ai', recursive=True)
assert ai_dirs, 'No */ai folders found. Add a GenImage-style dataset to the notebook inputs.'

gens, roots, splits = set(), set(), set()
for d in ai_dirs:
    split = os.path.basename(os.path.dirname(d))            # e.g. train / val
    gen   = os.path.basename(os.path.dirname(os.path.dirname(d)))
    root  = os.path.dirname(os.path.dirname(os.path.dirname(d)))
    if os.path.isdir(os.path.join(os.path.dirname(d), 'nature')):  # has reals too
        gens.add(gen); roots.add(root); splits.add(split)

assert len(roots) == 1, f'ambiguous roots: {roots}'
GENIMAGE_ROOT = roots.pop()
SPLIT = 'train' if 'train' in splits else sorted(splits)[0]
ALL_GENS = sorted(gens)
GENERATORS = ALL_GENS               # use ALL generators for the full LOGO table
PER_CLASS_LIMIT = 2000              # per generator per class; raise once happy

print(f'\nGENIMAGE_ROOT = {GENIMAGE_ROOT}')
print(f'SPLIT         = {SPLIT}  (available: {sorted(splits)})')
print(f'generators present ({len(ALL_GENS)}): {ALL_GENS}')
print(f'using GENERATORS = {GENERATORS}')

In [ ]:
# 3) Extraction pass: writes one npz per backbone into feature_cache/ (tag=train).
#    First run downloads CLIP-L/14 (~1.7GB) + DINOv2-base; needs Internet ON.
!python scripts/01_extract_features.py \
    --genimage-root {GENIMAGE_ROOT} \
    --generators {' '.join(GENERATORS)} \
    --per-class-limit {PER_CLASS_LIMIT} \
    --split {SPLIT} --tag train --device cuda

In [ ]:
# 4) Sanity-check the caches: shapes, label balance, alignment across backbones.
from pmsa.features import FeatureSet
import numpy as np, glob
sets = {}
for f in sorted(glob.glob('feature_cache/*_train.npz')):
    fs = FeatureSet.load(f); sets[fs.backbone] = fs
    print(f'{fs.backbone:10s} feats={fs.features.shape} '
          f'real={int((fs.labels==0).sum())} fake={int((fs.labels==1).sum())} '
          f'gens={sorted(set(fs.source[fs.labels==1]))}')
names = list(sets)
if len(names) > 1:
    base = sets[names[0]]
    aligned = all(np.array_equal(sets[n].paths, base.paths) for n in names[1:])
    print('caches aligned across backbones:', aligned)

In [ ]:
# 5) The floor: UnivFD linear probe on frozen CLIP, leave-one-generator-out.
#    Each held-out generator is the zero-shot test. This is the number v2 must beat.
!python scripts/02_train_baseline.py --backbone clip_l14 --tag train
import json
r = json.load(open('outputs/baseline_clip_l14_logo.json'))
for g, rep in r.items():
    print(f"held-out {g:24s} AUC={rep['auc']['point']:.4f} "
          f"[{rep['auc']['lo']:.3f},{rep['auc']['hi']:.3f}]  "
          f"TPR@1%={rep['tpr_at_0.01']['point']:.4f}")

In [ ]:
# 6) PMSA v2 fusion (CLIP + DINOv2 + NPR), full LOGO, multi-seed.
#    Compare on TPR@1% — the Neyman-Pearson operating point — not just AUC.
!python scripts/03_train_fusion.py --tag train

import json, numpy as np
base = json.load(open('outputs/baseline_clip_l14_logo.json'))
fus  = json.load(open('outputs/fusion_logo.json'))

def mean_pt(seedmap, key):  # mean over seeds (fusion stores seed->report)
    return float(np.mean([v[key]['point'] for v in seedmap.values()]))

print(f"{'held-out':30s}{'base AUC':>10}{'fus AUC':>10} | "
      f"{'base TPR@1%':>12}{'fus TPR@1%':>12}{'Δ':>9}")
print('-'*86)
for g in sorted(fus):
    bA, fA = base[g]['auc']['point'], mean_pt(fus[g], 'auc')
    bT, fT = base[g]['tpr_at_0.01']['point'], mean_pt(fus[g], 'tpr_at_0.01')
    flag = '  <-- worse' if fT < bT - 0.005 else ''
    print(f'{g:30s}{bA:>10.4f}{fA:>10.4f} | {bT:>12.4f}{fT:>12.4f}{fT-bT:>+9.4f}{flag}')

## 7) The contribution: FPR guarantee under real-domain shift

Everything above is the **generator-shift** column (real domain fixed at ImageNet).
Now the actual thesis: add a **second real domain** (FFHQ faces) and show that the
Neyman-Pearson false-alarm guarantee (FPR ≤ α), calibrated on ImageNet reals,
**breaks** under the new real domain — then is **restored** by recalibrating on
FFHQ reals *only* (no fakes needed).

**Add an FFHQ dataset to the notebook inputs first**, e.g.
[`arnaud58/flickrfaceshq-dataset-ffhq`](https://www.kaggle.com/datasets/arnaud58/flickrfaceshq-dataset-ffhq)
(or any folder of face images). Then set `FFHQ_DIR` below to its path.

In [ ]:
# 8) Extract FFHQ reals into the same feature space (tag=ffhq, reals only).
import glob, os
# auto-find an FFHQ-looking input folder, or set FFHQ_DIR manually
cands = [p for p in glob.glob('/kaggle/input/*') if 'ffhq' in p.lower() or 'face' in p.lower()]
FFHQ_DIR = cands[0] if cands else '/kaggle/input/flickrfaceshq-dataset-ffhq'  # <-- EDIT if needed
print('FFHQ_DIR =', FFHQ_DIR, '| exists:', os.path.isdir(FFHQ_DIR))

!python scripts/01_extract_features.py \
    --real-dir ffhq:{FFHQ_DIR} \
    --real-limit 2000 \
    --tag ffhq --device cuda

In [ ]:
# 8b) Extract FAKE faces (StyleGAN2) into the same feature space (tag=facesfake).
#     Add a fake-faces dataset first, e.g. 'hyperclaw79/fakefaces' (StyleGAN2 1024^2).
import glob, os
fcands = [p for p in glob.glob('/kaggle/input/*')
          if 'fake' in p.lower() and ('face' in p.lower() or 'gan' in p.lower())]
FAKEFACE_DIR = fcands[0] if fcands else '/kaggle/input/fakefaces'  # <-- EDIT if needed
print('FAKEFACE_DIR =', FAKEFACE_DIR, '| exists:', os.path.isdir(FAKEFACE_DIR))

!python scripts/01_extract_features.py \
    --fake-dir stylegan2:{FAKEFACE_DIR} \
    --fake-limit 2000 \
    --tag facesfake --device cuda

In [ ]:
# 9) Run the full real-domain-shift experiment: FPR break/restore + power cost.
!python scripts/06_real_domain_shift.py \
    --train-tag train --shift-tag ffhq --shift-fake-tag facesfake \
    --base-domain imagenet --shift-domain ffhq

# What to look for:
#   FPR:  in-domain ≈ alpha | shifted >> alpha (breaks) | recalibrated ≈ alpha (restored)
#   TPR:  in-domain high | new-domain @ base tau | new-domain @ recal tau
#     If TPR @ recal tau collapses vs @ base tau  -> "FPR recoverable, power is NOT".
#     If it stays high                            -> recalibration restores deployable use.
#   face_auc = can the detector even separate real vs fake faces it never trained on?

In [ ]:
# 10) Few-shot power recovery: how many labeled face fakes restore TPR?
#     FPR is pinned at alpha (recalibrated on face reals) at every budget k.
!python scripts/07_few_shot_recovery.py \
    --train-tag train --shift-tag ffhq --shift-fake-tag facesfake \
    --k-list 0,10,25,50,100,250,500,1000

# Read the curve:
#   k=0   = reals-only adaptation (no labeled fakes) -> the power floor
#   TPR should climb with k while FPR stays ~alpha (guarantee held throughout)
#   The k where TPR plateaus = the labeled-fake budget that buys back power.

In [ ]:
# 11) Generality: add COCO as a MILD real-domain shift (objects, near ImageNet).
#     Add 'xthink/coco-2017-val-images' to inputs first. Reals only.
import glob, os
ccands = [p for p in glob.glob('/kaggle/input/**', recursive=True)
          if os.path.isdir(p) and 'coco' in p.lower() and 'val' in p.lower()]
COCO_DIR = ccands[0] if ccands else '/kaggle/input/coco-2017-val-images'  # <-- EDIT if needed
print('COCO_DIR =', COCO_DIR, '| exists:', os.path.isdir(COCO_DIR))

!python scripts/01_extract_features.py \
    --real-dir coco:{COCO_DIR} \
    --real-limit 2000 \
    --tag coco --device cuda

In [ ]:
# 12) The severity gradient: how the FPR break scales with real-domain distance.
#     One detector, one base tau; measured across imagenet (base) -> coco -> ffhq.
#     Add chameleon to the list once you have it (it's gated; email the AIDE authors).
!python scripts/08_shift_generality.py \
    --train-tag train --base-domain imagenet --shift-tags coco,ffhq

# Expect: FPR @ base tau grows with shift severity (imagenet ~0.01 < coco < ffhq ~0.26),
#         while FPR @ recal stays ~alpha for every domain (guarantee restorable everywhere).

In [4]:
!python scripts/build_manifest.py \
  --genimage-root /kaggle/input/datasets/yangsangtai/tiny-genimage \
  --ffhq-dir /kaggle/input/datasets/arnaud58/flickrfaceshq-dataset-ffhq \
  --fakefaces-dir /kaggle/input/datasets/hyperclaw79/fakefaces \
  --coco-dir /kaggle/input/datasets/xthink/coco-2017-val-images/val2017 \
  --chameleon-dir /kaggle/input/datasets/yashyadav56/chameleon-private/Chameleon/test \
  --per-generator 2000 --per-source 6000 --balance

!python scripts/01_extract_features.py --manifest feature_cache/manifest_diverse_train.csv \
  --tag diversetrain --backbones siglip forensic --normalize-jpeg --device cuda
!python scripts/01_extract_features.py --manifest feature_cache/manifest_chameleon_test.csv \
  --tag chameleontest --backbones siglip forensic --normalize-jpeg --device cuda

genimage: 28000 (generators=['imagenet_ai_0419_biggan', 'imagenet_ai_0419_vqdm', 'imagenet_ai_0424_sdv5', 'imagenet_ai_0424_wukong', 'imagenet_ai_0508_adm', 'imagenet_glide', 'imagenet_midjourney'])
ffhq real: 6000
stylegan2 fake: 6000
coco real: 5000

[TRAIN] 40000 images  | real=20000 fake=20000
  by domain: {'faces': 10737, 'imagenet': 25249, 'coco': 4014}
  by source: {'real': 20000, 'stylegan2': 6000, 'imagenet_ai_0508_adm': 2000, 'imagenet_glide': 2000, 'imagenet_ai_0424_sdv5': 2000, 'imagenet_ai_0419_vqdm': 2000, 'imagenet_midjourney': 2000, 'imagenet_ai_0419_biggan': 2000, 'imagenet_ai_0424_wukong': 2000}
-> feature_cache/manifest_diverse_train.csv

[TEST (Chameleon, held out)] 26033 images  | real=14863 fake=11170
  by domain: {'chameleon': 26033}
  by source: {'real': 14863, 'chameleon': 11170}
-> feature_cache/manifest_chameleon_test.csv
manifest: 40000 images from feature_cache/manifest_diverse_train.csv
config.json: 100%|█████████████████████████████| 432/432 [00:00<00:00,

In [5]:
!python scripts/train_deploy.py --model probe --backbone siglip --tag diversetrain --alpha 0.05
!python scripts/eval_testset.py --model probe --backbone siglip --test-tag chameleontest
!python scripts/train_deploy.py --model fusion --backbones siglip forensic --tag diversetrain --alpha 0.05
!python scripts/eval_testset.py --model fusion --test-tag chameleontest


[probe] held-out: AUC=0.9747 TPR@1%=0.6101 TPR@5%=0.8944
saved model -> outputs/deploy/probe.pkl
saved calibrator (alpha=0.05, tau=0.646) -> outputs/deploy/calibrator.json
ready for: PMSA_CKPT=outputs/deploy/probe.pkl python app.py

=== chameleontest [probe] (26033 imgs: 14863 real / 11170 fake) ===
AUC        0.8511 [0.847, 0.855]
TPR@1%FPR  0.1430
TPR@5%FPR  0.3755

At deployed tau=0.646 (alpha=0.05):
  realized FPR (false alarms on real) 0.0630
  detection rate (TPR on fake)        0.4153
  accuracy                            0.7131

(in-the-wild ~65-75% AUC = SOTA range = success; report this honestly)
epoch 0: val_loss=0.1694
epoch 10: val_loss=0.1372
epoch 20: val_loss=0.1488
epoch 30: val_loss=0.1900
epoch 40: val_loss=0.1960

[fusion] held-out: AUC=0.9932 TPR@1%=0.8847 TPR@5%=0.9724
saved model -> outputs/deploy/fusion.pt
saved calibrator (alpha=0.05, tau=-0.787) -> outputs/deploy/calibrator.json
ready for: PMSA_CKPT=outputs/deploy/fusion.pt python app.py

=== chameleontest [f

In [6]:
!cd feature_cache && zip -rq /kaggle/working/pmsa_features.zip *.npz *.csv
!ls -lh /kaggle/working/pmsa_features.zip

-rw-r--r-- 1 root root 199M Jun 16 14:06 /kaggle/working/pmsa_features.zip


In [5]:
#/kaggle/input/datasets/yashyadav56/pmssa-features
import glob, shutil, os
os.makedirs('feature_cache', exist_ok=True)
n = 0
for f in glob.glob('/kaggle/input/**/*.npz', recursive=True):
    shutil.copy(f, 'feature_cache/'); n += 1
print(f'copied {n} npz -> {os.path.abspath("feature_cache")}')
print(sorted(os.listdir('feature_cache')))

copied 4 npz -> /kaggle/working/pmsa_v2/feature_cache
['.gitkeep', 'forensic_chameleontest.npz', 'forensic_diversetrain.npz', 'siglip_chameleontest.npz', 'siglip_diversetrain.npz']


In [4]:
# extract GPT-Image features (just 100 imgs — fast)
!python scripts/01_extract_features.py \
  --fake-dir gptimage:/kaggle/input/datasets/yashyadav56/gpt-images \
  --tag gptimage --backbones siglip forensic --normalize-jpeg --device cuda


fake[gptimage]: 100 images
manifest: 100 total -> feature_cache/manifest_gptimage.csv
config.json: 100%|█████████████████████████████| 432/432 [00:00<00:00, 1.85MB/s]
model.safetensors: 100%|██████████████████████| 813M/813M [00:04<00:00, 168MB/s]
Loading weights: 100%|█| 208/208 [00:00<00:00, 1552.79it/s, Materializing param=
SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn

In [5]:
# the experiment: zero-shot -> few-shot curve
!python scripts/09_frontier_fewshot.py \
  --train-tag diversetrain --new-fake-tag gptimage \
  --backbones siglip forensic --k-list 0,10,25,50 --new-name GPT-Imag


=== few-shot adaptation to GPT-Imag (100 fakes) | alpha=0.01 seeds=[0, 1, 2] ===
 k (labeled)   detection (TPR)      FPR      AUC
------------------------------------------------
           0      0.300+/-0.028    0.011    0.903  <- zero-shot
          10      0.673+/-0.034    0.012    0.951
          25      0.787+/-0.034    0.010    0.974
          50      0.827+/-0.009    0.012    0.985

FPR stays ~alpha at every k; TPR is the recovery from zero-shot.
saved -> outputs/frontier_fewshot_gptimage.json


In [6]:
!python scripts/01_extract_features.py \
  --fake-dir nanobanana:/kaggle/input/datasets/yashyadav56/nano-images \
  --tag nanobanana --backbones siglip forensic --normalize-jpeg --device cuda

fake[nanobanana]: 100 images
manifest: 100 total -> feature_cache/manifest_nanobanana.csv
config.json: 100%|█████████████████████████████| 432/432 [00:00<00:00, 2.46MB/s]
model.safetensors: 100%|██████████████████████| 813M/813M [00:04<00:00, 168MB/s]
Loading weights: 100%|█| 208/208 [00:00<00:00, 1501.32it/s, Materializing param=
SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_

In [7]:
!python scripts/09_frontier_fewshot.py \
  --train-tag diversetrain --new-fake-tag nanobanana \
  --backbones siglip forensic --k-list 0,10,25,50 --new-name "Nano Banana"


=== few-shot adaptation to Nano Banana (100 fakes) | alpha=0.01 seeds=[0, 1, 2] ===
 k (labeled)   detection (TPR)      FPR      AUC
------------------------------------------------
           0      0.573+/-0.141    0.011    0.963  <- zero-shot
          10      0.793+/-0.115    0.010    0.976
          25      0.900+/-0.033    0.011    0.989
          50      0.907+/-0.034    0.011    0.986

FPR stays ~alpha at every k; TPR is the recovery from zero-shot.
saved -> outputs/frontier_fewshot_nanobanana.json


## What to look for

- **Caches aligned = True** and equal row counts per backbone — if not, the
  dataloader order broke; fusion can't fuse misaligned streams.
- **LOGO AUC** well above 0.5 means the frozen-CLIP baseline transfers to a held-out
  generator. With only 2 generators this is a weak test; the point here is that the
  *loop runs end to end*, not the final number.

## Next once green

1. Scale: `PER_CLASS_LIMIT = 10000`, add more generators, re-extract.
2. Extract a `--tag test` set that also includes a **second real domain**
   (`--real-dir ffhq:/path`) and in-the-wild fakes (`--fake-dir nano_banana:/path`)
   so the shift matrix (scripts 04–05) has cells C/D to fill.
3. Save `feature_cache/*.npz` as a Kaggle **dataset output** so you can train on the
   M4 without re-running extraction.
4. Phase 3: `scripts/03_train_fusion.py` (runs fine on the laptop from the caches).